In [1]:
import torch

print("=" * 50)
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())
print("=" * 50)

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Name: Tesla T4
GPU Count: 1


In [2]:
!pip install -q \
transformers==4.48.3 \
accelerate==1.3.0 \
datasets==3.2.0 \
peft==0.14.0 \
trl==0.14.0 \
bitsandbytes==0.45.2

In [3]:
from huggingface_hub import login

login()


In [4]:
from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("✅ Tokenizer loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

✅ Tokenizer loaded successfully!


In [5]:
from datasets import load_dataset

dataset = load_dataset("acomquest/Saamayik")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.80M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/328k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/331k [00:00<?, ?B/s]

data/test_ood-00000-of-00001.parquet:   0%|          | 0.00/714k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43493 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2416 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2417 [00:00<?, ? examples/s]

Generating test_ood split:   0%|          | 0/4047 [00:00<?, ? examples/s]

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['translation', 'source'],
        num_rows: 43493
    })
    validation: Dataset({
        features: ['translation', 'source'],
        num_rows: 2416
    })
    test: Dataset({
        features: ['translation', 'source'],
        num_rows: 2417
    })
    test_ood: Dataset({
        features: ['translation', 'source'],
        num_rows: 4047
    })
})


In [7]:
print(dataset["train"][0])

{'translation': {'en': '"Save it with Ctrl, S."', 'sa': '"Ctrl, S नुत्वा रक्षन्तु।"'}, 'source': 'main'}


In [8]:
for i in range(5):
    print("="*80)
    print("Sanskrit:")
    print(dataset["train"][i]["translation"]["sa"])
    print()
    print("English:")
    print(dataset["train"][i]["translation"]["en"])

Sanskrit:
"Ctrl, S नुत्वा रक्षन्तु।"

English:
"Save it with Ctrl, S."
Sanskrit:
गुरुः छात्रान् एकवारं पाठयति ।

English:
Teacher will teach the students only once.
Sanskrit:
चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित्रद्वयं स्वीकरणीयमस्ति । इदमतीव सुलभमस्ति ।

English:
"To recreate this animation, I have to take two images out of this stack which is very easy."
Sanskrit:
वयं  Colors विकल्पं तस्योपरि नोदनेन चिनुमः ।

English:
I will choose Colors options by clicking on it.
Sanskrit:
"""अत्र कानिचन उदाहरणानि पश्याम:- एक: पर्वत:, चत्वारो ग्रामा:, चत्वारो मार्गा: ये लक्ष्यं (पताकां) प्रति गच्छन्ति।"""

English:
"""See the example here - one mountain, four villages, four paths leading to one goal towards flag."""


In [9]:
import pandas as pd

train_df = pd.DataFrame({
    "sanskrit": [x["translation"]["sa"] for x in dataset["train"]],
    "english": [x["translation"]["en"] for x in dataset["train"]]
})

train_df.head()

,sanskrit,english
0,"""Ctrl, S नुत्वा रक्षन्तु।""","""Save it with Ctrl, S."""
1,गुरुः छात्रान् एकवारं पाठयति ।,Teacher will teach the students only once.
2,चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...,"""To recreate this animation, I have to take tw..."
3,वयं Colors विकल्पं तस्योपरि नोदनेन चिनुमः ।,I will choose Colors options by clicking on it.
4,"""""""अत्र कानिचन उदाहरणानि पश्याम:- एक: पर्वत:, ...","""""""See the example here - one mountain, four v..."


In [10]:
print(train_df.isnull().sum())

sanskrit    0
english     0
dtype: int64


In [11]:
train_df = train_df.dropna()

In [12]:
train_df = train_df.drop_duplicates()

In [13]:
print("Total examples:", len(train_df))

Total examples: 43407


In [14]:
def create_prompt(row):
    return {
        "text": f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Translate the following Sanskrit sentence into English.

Sanskrit:
{row['sanskrit']}

<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{row['english']}<|eot_id|>"""
    }

instruction_df = train_df.apply(create_prompt, axis=1, result_type="expand")
instruction_df.head()

,text
0,<|begin_of_text|><|start_header_id|>user<|end_...
1,<|begin_of_text|><|start_header_id|>user<|end_...
2,<|begin_of_text|><|start_header_id|>user<|end_...
3,<|begin_of_text|><|start_header_id|>user<|end_...
4,<|begin_of_text|><|start_header_id|>user<|end_...


In [15]:
print(instruction_df.iloc[0]["text"])

<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Translate the following Sanskrit sentence into English.

Sanskrit:
"Ctrl, S नुत्वा रक्षन्तु।"

<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"Save it with Ctrl, S."<|eot_id|>


In [16]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [17]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [18]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [19]:
tokenizer.pad_token = tokenizer.eos_token

In [20]:
import transformers
import trl
import peft
import datasets
import torch

print(transformers.__version__)
print(trl.__version__)
print(peft.__version__)
print(datasets.__version__)
print(torch.__version__)

4.48.3
0.14.0
0.14.0
3.2.0
2.11.0+cu128


In [22]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ Model loaded successfully!")

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Model loaded successfully!


In [23]:
model = prepare_model_for_kbit_training(model)

In [24]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [25]:
model = get_peft_model(model, lora_config)

In [26]:
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


In [57]:
from datasets import Dataset

hf_dataset = Dataset.from_pandas(instruction_df)

print(hf_dataset)

Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 43407
})


In [58]:
hf_dataset = hf_dataset.remove_columns(["__index_level_0__"])

In [59]:
print(hf_dataset)

Dataset({
    features: ['text'],
    num_rows: 43407
})


In [60]:
MAX_LENGTH = 256

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [61]:
tokenized_dataset = hf_dataset.map(
    tokenize,
    batched=False,
    remove_columns=["text"]
)

Map:   0%|          | 0/43407 [00:00<?, ? examples/s]

In [62]:
print(tokenized_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 43407
})


In [63]:
split = tokenized_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split["train"]
eval_dataset = split["test"]

In [64]:
print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 39066
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4341
})


In [65]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [66]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama_sanskrit_lora",

    overwrite_output_dir=True,

    num_train_epochs=1,

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    weight_decay=0.01,

    logging_steps=20,

    save_strategy="epoch",

    eval_strategy="epoch",

    fp16=True,

    report_to="none"
)

In [67]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

/tmp/ipykernel_4114/2557268516.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [68]:
print(type(trainer))

<class 'transformers.trainer.Trainer'>


In [69]:
trainer.train()

Epoch,Training Loss,Validation Loss
0,1.565800,1.541765


TrainOutput(global_step=2441, training_loss=1.7642842515576622, metrics={'train_runtime': 4790.812, 'train_samples_per_second': 8.154, 'train_steps_per_second': 0.51, 'total_flos': 5.858348912502374e+16, 'train_loss': 1.7642842515576622, 'epoch': 0.999744022935545})

In [70]:
trainer.save_model("./llama_sanskrit_lora")
tokenizer.save_pretrained("./llama_sanskrit_lora")

('./llama_sanskrit_lora/tokenizer_config.json',
 './llama_sanskrit_lora/special_tokens_map.json',
 './llama_sanskrit_lora/tokenizer.json')

In [71]:
import os

print(os.listdir("./llama_sanskrit_lora"))

['tokenizer_config.json', 'README.md', 'checkpoint-2441', 'adapter_model.safetensors', 'training_args.bin', 'tokenizer.json', 'adapter_config.json', 'special_tokens_map.json']


In [72]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "./llama_sanskrit_lora"
)

model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [73]:
print(
    translate("अहं विद्यालयं गच्छामि।")
)

NameError: name 'translate' is not defined

In [74]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [75]:
print(tokenizer)

PreTrainedTokenizerFast(name_or_path='meta-llama/Llama-3.2-1B-Instruct', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>', 'pad_token': '<|eot_id|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|res

In [76]:
def translate(sentence):

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Translate the following Sanskrit sentence into English.

Sanskrit:
{sentence}

<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.2,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [77]:
print(translate("अहं विद्यालयं गच्छामि।"))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


user

Translate the following Sanskrit sentence into English.

Sanskrit:
अहं विद्यालयं गच्छामि।

assistant
I will go to the school.


In [78]:
test_examples = instruction_df.sample(10, random_state=42)

for i, row in test_examples.iterrows():
    prompt = row["text"]

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        temperature=0.2
    )

    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("=" * 80)
    print(f"Example {i}")
    print(prediction)
    print("=" * 80)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Example 25797
user

Translate the following Sanskrit sentence into English.

Sanskrit:
अधुना main फ़ङ्क्षन् विषयम् अधिकं ज्ञातुं स्लैड्स् प्रति गच्छामः ।

assistant

Now  Let us switch to the slides to know more about  main function. Let us go to the next slide.assistant

Let us switch to the next slide. Now we will learn about main function. Let us go to the next slide.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Example 22562
user

Translate the following Sanskrit sentence into English.

Sanskrit:
नदी प्रवहति ।

assistant

River flows.assistant

River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows. River flows


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Example 5346
user

Translate the following Sanskrit sentence into English.

Sanskrit:
अध्यापिका प्रश्नं पृच्छति ।

assistant

Teacher asks question.assistant

Teacher asks question.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Example 7140
user

Translate the following Sanskrit sentence into English.

Sanskrit:
ईक्वल् टु उदाहरणार्थं ए ईक्वल् टू बि a== b

assistant

Equal to: e.g. a==bassistant

Equal to: e.g. a==b.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Example 25074
user

Translate the following Sanskrit sentence into English.

Sanskrit:
अतः &num इतीदं num इत्यस्य मेमोरि अड्रेस् यच्छति ।

assistant

"So, ampersand num (&num) will give the memory address of num."assistant

"So, ampersand num (&num) will give the memory address of num."
Example 5307
user

Translate the following Sanskrit sentence into English.

Sanskrit:
तेषु नौकां वाहयत्सु स निदद्रौ;

assistant

"""But as they sailed he fell asleep: and there came down a storm of wind on the lake; and they were filled with water, and were in jeopardy."""assistant

"""And he fell asleep at the helm, and the storm was upon them:""""""And they were in great fear, and did not know the way."""


In [79]:
!pip install -q sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00


In [80]:
from sacrebleu import corpus_bleu

In [82]:
print(instruction_df.columns)

Index(['text'], dtype='object')


In [83]:
import pandas as pd
import re

results = []

sample = instruction_df.sample(10, random_state=42)

for _, row in sample.iterrows():

    prompt = row["text"]

    # Extract Sanskrit sentence
    sanskrit = re.search(
        r"Sanskrit:\n(.*?)\n\n<\|eot_id\|>",
        prompt,
        re.DOTALL
    ).group(1).strip()

    # Extract ground truth English
    reference = re.search(
        r"<\|start_header_id\|>assistant<\|end_header_id\|>\n\n(.*?)<\|eot_id\|>",
        prompt,
        re.DOTALL
    ).group(1).strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Keep only the assistant's generated answer
    if "assistant" in prediction:
        prediction = prediction.split("assistant")[-1].strip()

    results.append({
        "Sanskrit": sanskrit,
        "Ground Truth": reference,
        "Prediction": prediction
    })

results_df = pd.DataFrame(results)

results_df

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


,Sanskrit,Ground Truth,Prediction
0,अधुना main फ़ङ्क्षन् विषयम् अधिकं ज्ञातुं स्लै...,Now Let us switch to the slides to know more ...,Let us switch to the next slide. Now we will l...
1,नदी प्रवहति ।,River flows.,River flows. River flows. River flows. River f...
2,यदा भवान् चित्रगवाक्षे (image window) प्लवनकील...,"""When you press tab into the image window, the...","""When you press tab into the image window, the..."
3,प्रथमतया IDE इतीदं संस्थापयितुं यतामहै।,Let us first look at installing the IDE.,Let us first look at installing the IDE.
4,"""वामपार्श्वे Course category मध्ये, 1st year M...",Click on 1st year Maths in the Course category...,Click on 1st year Maths in the Course category...
5,सञ्चिकां रक्षन्तु।,Save the file.,Save the file.
6,अध्यापिका प्रश्नं पृच्छति ।,Teacher asks question.,Teacher asks question.
7,ईक्वल् टु उदाहरणार्थं ए ईक्वल् टू बि a== b,Equal to: e.g. a==b,Equal to: e.g. a==b.
8,अतः &num इतीदं num इत्यस्य मेमोरि अड्रेस् यच्छ...,"""So, ampersand num (&num) will give the memory...","""So, ampersand num (&num) will give the memory..."
9,तेषु नौकां वाहयत्सु स निदद्रौ;,"""""""But as they sailed he fell asleep: and ther...","""""""And he fell asleep at the helm, and the sto..."
